# Training Tricks — Exercises

Companion to the note [Training Tricks](Training%20Tricks.md).

Practise the note's toolbox: initialisation (Xavier/He, never zeros), normalisation (BatchNorm, LayerNorm, RMSNorm),
regularisation (dropout, label smoothing, mixup, early stopping), optimisation helpers (gradient clipping, gradient accumulation),
the debug checklist (initial loss $\approx\log K$, overfit a tiny batch) and transfer learning. Everything is implemented from scratch in NumPy,
with finite-difference gradient checks where backward passes are involved.

**17 exercises** · 🧠 Concept ×4 · ✍️ By hand ×6 · 💻 Code ×7

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

from scipy.special import softmax as _sp_softmax, log_softmax as _sp_log_softmax
from sklearn.datasets import load_digits
rng = np.random.default_rng(0)

def _num_grad(f, x, eps=1e-6):
    # reference central-difference gradient of scalar f() w.r.t. array x (modified in place and restored)
    g = np.zeros_like(x, dtype=float)
    it = np.nditer(x, flags=['multi_index'])
    for _ in it:
        i = it.multi_index
        old = x[i]
        x[i] = old + eps; fp = f()
        x[i] = old - eps; fm = f()
        x[i] = old
        g[i] = (fp - fm) / (2 * eps)
    return g

# A small ready-made MLP (1 tanh hidden layer, softmax output) used in Part D
def mlp_init(d, h, k, seed=0):
    r = np.random.default_rng(seed)
    return [r.normal(0, 1 / np.sqrt(d), (d, h)), np.zeros(h), r.normal(0, 0.01, (h, k)), np.zeros(k)]

def mlp_loss_grads(params, X, y):
    W1, b1, W2, b2 = params
    N = len(y)
    H = np.tanh(X @ W1 + b1)
    Z = H @ W2 + b2
    loss = -_sp_log_softmax(Z, axis=1)[np.arange(N), y].mean()
    D2 = _sp_softmax(Z, axis=1); D2[np.arange(N), y] -= 1; D2 /= N
    D1 = (D2 @ W2.T) * (1 - H ** 2)
    return loss, [X.T @ D1, D1.sum(0), H.T @ D2, D2.sum(0)]

## Part A · Concepts

### Exercise 1 · Why never initialise to zeros?
*🧠 Concept · ★☆☆*

1. Show that if all weights of an MLP's hidden layer start at the **same** value (e.g. 0), all hidden units stay identical forever under gradient descent.
2. What exactly goes wrong with all-zeros in a ReLU or tanh network (look at the gradients)?
3. Is it fine to initialise the **biases** to zero?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

If two hidden units have identical incoming and outgoing weights, are their activations, their $\delta$'s, and hence their gradients different?

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. Identical units compute identical activations $h_j$ and receive identical errors $\delta_j = (W^\top\delta)_j\phi'(z_j)$ (their outgoing weights are equal too),
   hence identical gradients and identical updates. By induction they stay identical: the layer behaves like a single unit (**symmetry is never broken**).
2. With all zeros: $z = 0$, so tanh outputs 0 and the gradient w.r.t. the first-layer weights $\delta^{(1)}x^\top$ vanishes because $\delta^{(1)} = W^{(2)\top}\delta^{(2)}\odot\phi' = 0$ ($W^{(2)}=0$).
   Only the last bias learns. ReLU additionally has $\phi'(0) = 0$ by convention.
3. Yes: random weights already break the symmetry, so zero biases are standard (sometimes a small positive bias for ReLU to avoid dead units).

</details>

### Exercise 2 · Which normalisation layer?
*🧠 Concept · ★★☆*

For an activation tensor of shape $(N, C, H, W)$ say over which axes BatchNorm, LayerNorm and GroupNorm compute their mean and variance,
and pick one for each situation:

(a) A ResNet for ImageNet with batch size 256. (b) A Transformer for text with variable sequence lengths. (c) Object detection with 2 high-resolution images per GPU.
(d) Why does BatchNorm behave differently at train and test time, and what goes wrong if you forget `model.eval()`?
(e) What does RMSNorm drop compared with LayerNorm?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

BatchNorm couples the examples in a batch; LayerNorm and GroupNorm normalise each example on its own.

</details>

<details>
<summary><b>✅ Solution</b></summary>

BatchNorm: per channel over $(N, H, W)$. LayerNorm: per example over its features, e.g. $(C,H,W)$, or the last dimension $d$ for tokens. GroupNorm: per example over groups of channels and $(H,W)$.

(a) **BatchNorm**: large batches give good statistics and it is the standard in [[CNN]]s. (b) **LayerNorm** (or RMSNorm): independent of batch and sequence length,
no train/test discrepancy ([[Transformers]]). (c) **GroupNorm**: batch statistics from 2 images are far too noisy for BatchNorm.
(d) In training BN uses the current batch's statistics; at test time it uses running averages collected during training (predictions must not depend on the other test examples).
Forgetting `eval()` makes predictions depend on the test batch composition and degrades accuracy, especially for small or single-example batches.
(e) RMSNorm skips the mean subtraction (and usually the bias): $y = \gamma\,x/\sqrt{\overline{x^2}+\epsilon}$. Cheaper, and it works as well in LLMs.

</details>

### Exercise 3 · Debugging from symptoms
*🧠 Concept · ★★☆*

Using the note's debug checklist, give the most likely cause and the next step for each symptom (10-class classifier):

(a) The loss at step 0 is 23.0. (b) The loss stays at exactly 2.303 for 1000 steps. (c) The model cannot overfit 10 examples.
(d) The loss becomes `nan` after 3000 steps. (e) Training accuracy 99%, validation accuracy 60%. (f) Validation is better than training accuracy throughout.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

$\log 10 = 2.303$. Think: init scale, learning signal, bug in data/labels/loss, numerical blow-up, overfitting, regularisation only active in training.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) Initial loss should be $\approx\log 10 = 2.3$; 23 means very confident wrong logits: init scale too large (or labels wrong). Scale down the last layer's init.
(b) Stuck at $\log K$ = uniform prediction: no learning signal. Check that gradients are non-zero (dead ReLUs, detached graph, `optimizer.step()`/`zero_grad()` missing, lr = 0).
(c) A bug: labels misaligned with inputs, wrong loss/activation pairing (softmax applied twice), data augmentation or dropout too strong, or lr too small. This test must pass before anything else.
(d) Exploding gradients / too high lr / $\log 0$: watch gradient norms, add clipping, lower lr or add warm-up, use stable log-softmax / BCE-with-logits, check for bad data.
(e) Overfitting: more data/augmentation, weight decay, dropout, early stopping ([[Overfitting and Regularization]]).
(f) Not necessarily a bug: dropout/augmentation are active only during training, and training accuracy is often averaged over the epoch while the model was still improving.

</details>

### Exercise 4 · Transfer learning recipe
*🧠 Concept · ★☆☆*

You have an ImageNet-pretrained CNN and 2,000 labelled images of a new 5-class task. Describe the note's recipe step by step and explain
(a) why the backbone is frozen first, (b) why the learning rate is small when unfreezing, (c) what changes if you had 200,000 images, and (d) what to do with BatchNorm statistics when fine-tuning with tiny batches.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

A freshly initialised head produces large, random gradients. What would they do to good pretrained features?

</details>

<details>
<summary><b>✅ Solution</b></summary>

Replace the 1000-class head with a new 5-class head → **freeze** the backbone and train only the head → **unfreeze** (all or the top blocks) and fine-tune everything with a small lr (e.g. $10\times$ smaller, often with discriminative per-layer rates).
(a) The random head's large initial gradients would back-propagate into and destroy the pretrained features ("catastrophic forgetting"); training the head first makes them sensible.
(b) The pretrained weights are already good; large steps would move them far from that solution and overfit the 2,000 images.
(c) With lots of data you can unfreeze earlier/more layers and use a larger lr; the pretrained init mainly speeds up convergence.
(d) Keep BN layers in eval mode (frozen running statistics), since batch statistics from tiny batches are too noisy.

</details>

## Part B · By hand

### Exercise 5 · The initial loss should be log K
*✍️ By hand · ★☆☆*

At initialisation with small final-layer weights, the logits are $\approx 0$. What is the cross-entropy loss then, for $K = 10$ and for $K = 1000$ classes?
What is the initial BCE loss of a binary classifier whose output is $\sigma(0)$?

In [ ]:
loss_K10 = None
loss_K1000 = None
loss_binary = None

check('K = 10', loss_K10, -_sp_log_softmax(np.zeros(10))[0], tol=1e-3)
check('K = 1000', loss_K1000, -_sp_log_softmax(np.zeros(1000))[0], tol=1e-3)
check('binary', loss_binary, -np.log(0.5), tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

All logits equal → softmax is uniform: $\hat p_y = 1/K$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$L = -\log(1/K) = \log K$: $\log 10 = 2.303$, $\log1000 = 6.908$. Binary: $-\log0.5 = \log 2 = 0.693$.
If your first logged loss differs a lot from these, the init (or the loss) is off: checklist item 2.

```python
loss_K10 = 2.3026
loss_K1000 = 6.9078
loss_binary = 0.6931
```

</details>

### Exercise 6 · Deriving He initialisation
*✍️ By hand · ★★☆*

For $z_j = \sum_{i=1}^{n}W_{ji}h_i$ with independent zero-mean $W_{ji}$ (variance $s^2$) independent of the inputs:

1. Show $\mathrm{Var}(z_j) = n\,s^2\,\mathbb E[h_i^2]$.
2. For ReLU, $h = \max(0, z_{\text{prev}})$ with $z_{\text{prev}}$ symmetric around 0, show $\mathbb E[h^2] = \tfrac12\mathrm{Var}(z_{\text{prev}})$, and deduce He's choice $s^2 = 2/n$.
3. Compute the He standard deviation for $n = 512$ and the Xavier/Glorot-**uniform** limit $\sqrt{6/(n_{in}+n_{out})}$ for a $256\to128$ layer.

In [ ]:
he_std_512 = None
xavier_limit = None

check('He std', he_std_512, np.sqrt(2 / 512))
check('Xavier uniform limit (variance 2/(n_in+n_out))', xavier_limit, np.sqrt(3 * 2 / (256 + 128)))

<details>
<summary><b>💡 Hint</b></summary>

$\mathrm{Var}(Wh) = \mathbb E[W^2]\mathbb E[h^2] - (\mathbb E W\,\mathbb E h)^2$ and $\mathbb EW = 0$. For a symmetric $z$, $z^2\mathbb 1[z>0]$ has half the mass of $z^2$. A uniform on $[-a,a]$ has variance $a^2/3$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. Terms are independent with zero mean, so variances add: $\mathrm{Var}(z_j) = \sum_i\mathbb E[W_{ji}^2]\mathbb E[h_i^2] = n s^2\mathbb E[h^2]$.
2. $\mathbb E[h^2] = \mathbb E[z^2\mathbb 1[z>0]] = \tfrac12\mathbb E[z^2] = \tfrac12\mathrm{Var}(z)$. Keeping $\mathrm{Var}(z)$ constant across layers requires $n s^2\cdot\tfrac12 = 1$, i.e. $s^2 = 2/n$.
   (For tanh, linear near 0, $\mathbb E[h^2]\approx\mathrm{Var}(z)$ gives $s^2 = 1/n$; Xavier averages forward and backward: $2/(n_{in}+n_{out})$.)
3. He: $\sqrt{2/512} = 0.0625$. Xavier uniform: $a^2/3 = 2/384 \Rightarrow a = \sqrt{6/384} = 0.125$.

```python
he_std_512 = 0.0625
xavier_limit = 0.125
```

</details>

### Exercise 7 · Inverted dropout
*✍️ By hand · ★☆☆*

Inverted dropout with drop probability $p$ multiplies each activation by $m/(1-p)$, $m\sim\mathrm{Bernoulli}(1-p)$, during training and does nothing at test time.

1. Show $\mathbb E[\tilde h] = h$.
2. For $p = 0.2$: what is the scaling factor, and what is $\mathrm{Var}(\tilde h)$ for a fixed activation $h = 1$?

In [ ]:
scale = None
var_dropout = None

_m = np.random.default_rng(1).random(2_000_000) > 0.2
_ht = _m / 0.8
check('scale', scale, 1 / (1 - 0.2))
check('variance', var_dropout, _ht.var(), tol=5e-3)

<details>
<summary><b>💡 Hint</b></summary>

$\tilde h\in\{0, h/(1-p)\}$ with probabilities $p$, $1-p$. $\mathrm{Var} = \mathbb E[\tilde h^2] - (\mathbb E\tilde h)^2$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. $\mathbb E[\tilde h] = (1-p)\cdot\frac{h}{1-p} = h$, so test-time activations need no rescaling.
2. Scale $1/0.8 = 1.25$. $\mathbb E[\tilde h^2] = 0.8\cdot1.25^2 = 1.25$, so $\mathrm{Var} = 1.25 - 1 = 0.25 = p/(1-p)\cdot h^2$. The added multiplicative noise is the regulariser.

```python
scale = 1.25
var_dropout = 0.25
```

</details>

### Exercise 8 · Label smoothing
*✍️ By hand · ★☆☆*

With label smoothing $\varepsilon$, the target becomes $y^{LS} = (1-\varepsilon)\,y + \varepsilon/K$.

1. Write the smoothed target for $K = 5$, $\varepsilon = 0.1$, true class 0.
2. Why can the loss no longer be driven to 0, and what is its minimum possible value here (entropy of $y^{LS}$)?

In [ ]:
y_ls = None   # 5 numbers
min_loss = None

_y = np.eye(5)[0] * 0.9 + 0.1 / 5
check('smoothed target', y_ls, _y)
_z = np.log(_y)   # logits whose softmax equals the target achieve the minimum
check('minimum loss', min_loss, -(_y * _sp_log_softmax(_z)).sum(), tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

Cross-entropy $-\sum_k y^{LS}_k\log\hat p_k$ is minimised by $\hat p = y^{LS}$, giving the entropy $H(y^{LS})$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. $y^{LS} = (0.92, 0.02, 0.02, 0.02, 0.02)$.
2. Reaching 0 would need $\hat p_0 = 1$, but the target puts mass on every class; the minimum is $H(y^{LS}) = -0.92\log0.92 - 4\cdot0.02\log0.02 = 0.0767 + 0.3130 = 0.390$.
   The model is discouraged from pushing logits to $\pm\infty$: better calibration and a mild regulariser.

```python
y_ls = [0.92, 0.02, 0.02, 0.02, 0.02]
min_loss = 0.3897
```

</details>

### Exercise 9 · Gradient accumulation and effective batch size
*✍️ By hand · ★★☆*

You train on 4 GPUs (data parallel), each processing a micro-batch of 8, and accumulate gradients over 4 micro-steps before calling `optimizer.step()`.

1. What is the effective batch size?
2. Each micro-batch loss is a **mean** over its 8 examples. By what factor must you scale each micro-batch loss so the accumulated gradient equals the gradient of the mean loss over the effective batch?
3. Using the "linear scaling rule", if lr $=10^{-3}$ was tuned for batch 32, what lr would you try for the effective batch?

In [ ]:
eff_batch = None
loss_scale = None
lr_scaled = None

_g = np.random.default_rng(2).normal(size=(4 * 8 * 4, 3))   # per-example gradients
_acc = sum(0.25 * _g[i * 32:(i + 1) * 32].reshape(4, 8, 3).mean(1).mean(0) for i in range(4))   # 4 GPUs averaged, 4 accum steps scaled
check('effective batch', eff_batch, _g.shape[0])
check('loss scale', loss_scale, 0.25 if np.allclose(_acc, _g.mean(0)) else None)
check('linearly scaled lr', lr_scaled, 1e-3 * 128 / 32)

<details>
<summary><b>💡 Hint</b></summary>

Data-parallel training averages gradients across GPUs. Accumulation *sums* gradients across micro-steps.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. $4\times8\times4 = 128$.
2. The GPU all-reduce averages over $4\times8$ examples; summing 4 micro-steps would give $4\times$ the mean gradient, so scale each micro-loss by $1/4$ (the number of accumulation steps).
3. Linear scaling: $10^{-3}\cdot128/32 = 4\times10^{-3}$ (usually together with warm-up; for Adam a square-root rule is often safer).
Gradient accumulation trades time for memory: same gradient as a large batch (except for BatchNorm statistics, which still see 8 examples).

```python
eff_batch = 128
loss_scale = 0.25
lr_scaled = 4e-3
```

</details>

### Exercise 10 · Mixup by hand
*✍️ By hand · ★☆☆*

Mixup forms $\tilde x = \lambda x_i + (1-\lambda)x_j$ and $\tilde y = \lambda y_i + (1-\lambda)y_j$ with $\lambda\sim\mathrm{Beta}(\alpha,\alpha)$.
With $\lambda = 0.3$, $x_i = (1, 0, 2)$, class 0, $x_j = (0, 4, 2)$, class 2, $K = 3$: compute $\tilde x$ and $\tilde y$.
Then show that the cross-entropy against $\tilde y$ equals $\lambda\,\mathrm{CE}(\hat p, y_i) + (1-\lambda)\,\mathrm{CE}(\hat p, y_j)$.

In [ ]:
x_mix = None
y_mix = None

_lam = 0.3
check('mixed input', x_mix, _lam * np.array([1., 0, 2]) + (1 - _lam) * np.array([0., 4, 2]))
check('mixed label', y_mix, _lam * np.eye(3)[0] + (1 - _lam) * np.eye(3)[2])

<details>
<summary><b>💡 Hint</b></summary>

Cross-entropy is linear in the target distribution.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$\tilde x = (0.3, 2.8, 2.0)$, $\tilde y = (0.3, 0, 0.7)$.
$-\sum_k\tilde y_k\log\hat p_k = -\lambda\sum_k y_{i,k}\log\hat p_k - (1-\lambda)\sum_k y_{j,k}\log\hat p_k$, which is how mixup is implemented in practice (two CE terms, one forward pass).
Mixup encourages linear behaviour between training examples, improving calibration and robustness. CutMix pastes a patch instead and mixes labels by area.

```python
x_mix = [0.3, 2.8, 2.0]
y_mix = [0.3, 0, 0.7]
```

</details>

## Part C · Code: init and normalisation

### Exercise 11 · Initialisation and activation statistics
*💻 Code · ★★☆*

Push 1000 standard-normal inputs (width 256) through 20 ReLU layers (no biases) with three inits: `small` ($\mathcal N(0, 0.01^2)$),
`xavier` ($\mathcal N(0, 1/n)$) and `he` ($\mathcal N(0, 2/n)$). Store the standard deviation of the **last** layer's activations in `act_std` (dict).
This is checklist item 5, "watch activation statistics".

In [ ]:
act_std = None   # {'small': ..., 'xavier': ..., 'he': ...}

if act_std is None:
    check('He keeps the scale; small init collapses', None, True)
else:
    print({k: '%.2e' % v for k, v in act_std.items()})
    check('He keeps the scale; small init collapses', 0.3 < act_std['he'] < 3 and act_std['small'] < 1e-10 and act_std['xavier'] < act_std['he'] / 100, True)

<details>
<summary><b>💡 Hint</b></summary>

One loop over the three stds `{'small': 0.01, 'xavier': sqrt(1/256), 'he': sqrt(2/256)}`, same seed for each.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Small init: every layer multiplies the scale by $\approx 0.01\sqrt{256/2}\approx0.11$, so after 20 layers the activations are $\sim10^{-19}$ (gradients vanish too).
Xavier with ReLU halves the second moment each layer: std shrinks by $(1/\sqrt2)^{20}\approx10^{-3}$. He keeps it $\approx$ constant (std $\approx 1$, Exercise 6).

```python
act_std = {}
for name, s in {'small': 0.01, 'xavier': np.sqrt(1 / 256), 'he': np.sqrt(2 / 256)}.items():
    r = np.random.default_rng(0)
    H = r.normal(size=(1000, 256))
    for _ in range(20):
        H = np.maximum(0, H @ r.normal(0, s, (256, 256)))
    act_std[name] = H.std()
print(act_std)
```

</details>

### Exercise 12 · BatchNorm forward and backward
*💻 Code · ★★★*

Implement training-mode BatchNorm for a batch $X\in\mathbb R^{N\times D}$:
$\hat x = (x-\mu_B)/\sqrt{\sigma_B^2+\epsilon}$, $y = \gamma\hat x + \beta$ (statistics per feature over the batch, biased variance).
`bn_forward(X, gamma, beta, eps)` returns `(Y, cache)`; `bn_backward(dY, cache)` returns `(dX, dgamma, dbeta)`. The backward pass is checked by finite differences.

In [ ]:
def bn_forward(X, gamma, beta, eps=1e-5):
    return None, None

def bn_backward(dY, cache):
    return None, None, None

X_bn = rng.normal(3, 2, size=(6, 4)); g_bn = rng.normal(size=4); b_bn = rng.normal(size=4); G_bn = rng.normal(size=(6, 4))

_Y, _cache = bn_forward(X_bn, g_bn, b_bn)
if _Y is None:
    for _n in ('forward', 'dX', 'dgamma', 'dbeta'): check(_n, None, 0)
else:
    _Xh = (_Y - b_bn) / g_bn
    check('forward: x_hat has mean 0, var ~1 per feature', [_Xh.mean(0), _Xh.var(0)], [np.zeros(4), np.ones(4)], tol=1e-4)
    _dX, _dg, _db = bn_backward(G_bn, _cache)
    _L = lambda: np.sum(bn_forward(X_bn, g_bn, b_bn)[0] * G_bn)
    check('dX', _dX, _num_grad(_L, X_bn), tol=1e-6)
    check('dgamma', _dg, _num_grad(_L, g_bn), tol=1e-6)
    check('dbeta', _db, _num_grad(_L, b_bn), tol=1e-6)

<details>
<summary><b>💡 Hint 1</b></summary>

$d\beta = \sum_n dY_n$, $d\gamma = \sum_n dY_n\odot\hat x_n$, $d\hat x = dY\odot\gamma$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Compact form: $dX = \frac{1}{N\sqrt{\sigma^2+\epsilon}}\big(N\,d\hat x - \sum_n d\hat x_n - \hat x\odot\sum_n d\hat x_n\odot\hat x_n\big)$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Because $\mu_B$ and $\sigma_B$ depend on **every** example, the gradient for one example involves the whole batch: the two sums subtract the components of
$d\hat x$ along the directions that normalisation removes (constant shift and scaling). This coupling is also why BN behaves badly for tiny batches.
At test time use running averages $\mu, \sigma^2$ (updated as EMAs during training) instead of batch statistics.

```python
def bn_forward(X, gamma, beta, eps=1e-5):
    mu, var = X.mean(0), X.var(0)
    inv = 1 / np.sqrt(var + eps)
    Xh = (X - mu) * inv
    return gamma * Xh + beta, (Xh, inv, gamma)

def bn_backward(dY, cache):
    Xh, inv, gamma = cache
    N = dY.shape[0]
    dbeta = dY.sum(0)
    dgamma = (dY * Xh).sum(0)
    dXh = dY * gamma
    dX = inv / N * (N * dXh - dXh.sum(0) - Xh * (dXh * Xh).sum(0))
    return dX, dgamma, dbeta

X_bn = rng.normal(3, 2, size=(6, 4)); g_bn = rng.normal(size=4); b_bn = rng.normal(size=4); G_bn = rng.normal(size=(6, 4))
```

</details>

### Exercise 13 · LayerNorm and RMSNorm
*💻 Code · ★☆☆*

Implement `layer_norm(X, gamma, beta, eps=1e-5)` (normalise each **row** over its features) and `rms_norm(X, gamma, eps=1e-8)` ($x/\sqrt{\overline{x^2}+\epsilon}\cdot\gamma$).
Verify the key invariances: LayerNorm output is unchanged if a row is scaled and shifted ($x\to ax + c$, $a>0$); RMSNorm only under scaling.

In [ ]:
def layer_norm(X, gamma, beta, eps=1e-5):
    return None

def rms_norm(X, gamma, eps=1e-8):
    return None

X_ln = rng.normal(2, 3, size=(5, 8)); gam = np.ones(8); bet = np.zeros(8)

_Y = layer_norm(X_ln, gam, bet)
_R = rms_norm(X_ln, gam)
if _Y is None or _R is None:
    check('LayerNorm', None, 0); check('RMSNorm', None, 0)
else:
    check('LayerNorm rows: mean 0, var ~1', [_Y.mean(1), _Y.var(1)], [np.zeros(5), np.ones(5)], tol=1e-4)
    check('LayerNorm invariant to x -> 3x + 7', layer_norm(3 * X_ln + 7, gam, bet), _Y, tol=1e-4)
    check('RMSNorm rows have RMS 1', np.sqrt((_R ** 2).mean(1)), np.ones(5))
    check('RMSNorm invariant to x -> 3x', rms_norm(3 * X_ln, gam), _R)
    check('gamma/beta applied per feature', layer_norm(X_ln, 2 * gam, bet + 1), 2 * _Y + 1, tol=1e-4)

<details>
<summary><b>💡 Hint</b></summary>

Use `axis=1, keepdims=True` for the statistics. Same code at train and test time, no running averages needed.

</details>

<details>
<summary><b>✅ Solution</b></summary>

LayerNorm normalises each example independently, so it does not depend on the batch (no train/test discrepancy, works with batch size 1 and variable sequence lengths).
RMSNorm drops the centring; it is cheaper and is used in most modern LLMs. Neither is invariant to shifts in the way BatchNorm is across a batch.

```python
def layer_norm(X, gamma, beta, eps=1e-5):
    mu = X.mean(1, keepdims=True); var = X.var(1, keepdims=True)
    return gamma * (X - mu) / np.sqrt(var + eps) + beta

def rms_norm(X, gamma, eps=1e-8):
    return gamma * X / np.sqrt((X ** 2).mean(1, keepdims=True) + eps)

X_ln = rng.normal(2, 3, size=(5, 8)); gam = np.ones(8); bet = np.zeros(8)
```

</details>

## Part D · Code: regularisation and optimisation helpers

### Exercise 14 · Inverted dropout
*💻 Code · ★☆☆*

Implement `dropout(H, p, train, rng)`: in training mode zero each entry with probability `p` and scale survivors by $1/(1-p)$; in eval mode return `H` unchanged.
Also return the mask so the backward pass can reuse it (`dH = dY * mask`).

In [ ]:
def dropout(H, p, train, rng):
    # return (output, mask); mask already includes the 1/(1-p) scaling
    return None, None

H_do = np.ones((1000, 500))
out_train, mask_do = dropout(H_do, 0.3, True, np.random.default_rng(0))
out_eval, _ = dropout(H_do, 0.3, False, np.random.default_rng(0))

check('training mean preserved', None if out_train is None else out_train.mean(), 1.0, tol=0.01)
check('fraction dropped ~ p', None if out_train is None else np.mean(out_train == 0), 0.3, tol=0.01)
check('eval mode is identity', None if out_eval is None else out_eval, H_do)

<details>
<summary><b>💡 Hint</b></summary>

`mask = (rng.random(H.shape) >= p) / (1 - p)`; output `H * mask`. In eval mode return `H, np.ones_like(H)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Scaling during *training* ("inverted" dropout) means inference is just the plain network. Dropout regularises by preventing co-adaptation and approximates averaging an ensemble of thinned networks.
Remember to switch it off for evaluation (`model.eval()`), and to turn it off first while debugging (checklist item 4).

```python
def dropout(H, p, train, rng):
    if not train or p == 0:
        return H, np.ones_like(H)
    mask = (rng.random(H.shape) >= p) / (1 - p)
    return H * mask, mask

H_do = np.ones((1000, 500))
out_train, mask_do = dropout(H_do, 0.3, True, np.random.default_rng(0))
out_eval, _ = dropout(H_do, 0.3, False, np.random.default_rng(0))
```

</details>

### Exercise 15 · Gradient clipping by global norm
*💻 Code · ★☆☆*

Implement `clip_global_norm(grads, max_norm)` for a **list** of gradient arrays: if the global norm $\sqrt{\sum_i\lVert g_i\rVert^2}$ exceeds `max_norm`,
rescale **all** arrays by `max_norm / norm`; otherwise return them unchanged. Return `(clipped, norm_before)`.
Why is this better than clipping each element to $[-c, c]$?

In [ ]:
def clip_global_norm(grads, max_norm):
    return None, None

grads_big = [rng.normal(size=(3, 4)) * 10, rng.normal(size=4) * 10]
grads_small = [g / 1000 for g in grads_big]
clipped_big, norm_big = clip_global_norm(grads_big, 1.0)
clipped_small, _ = clip_global_norm(grads_small, 1.0)

_flat = lambda gs: np.concatenate([g.ravel() for g in gs])
check('norm before', norm_big, np.linalg.norm(_flat(grads_big)))
check('clipped norm = max_norm', None if clipped_big is None else np.linalg.norm(_flat(clipped_big)), 1.0)
check('direction preserved', None if clipped_big is None else _flat(clipped_big) / np.linalg.norm(_flat(clipped_big)), _flat(grads_big) / np.linalg.norm(_flat(grads_big)))
check('small gradients untouched', None if clipped_small is None else _flat(clipped_small), _flat(grads_small))

<details>
<summary><b>💡 Hint</b></summary>

`norm = np.sqrt(sum((g ** 2).sum() for g in grads))`; `scale = min(1, max_norm / (norm + 1e-12))`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Global-norm clipping only shrinks the step's **length**; the direction is still the true gradient direction. Element-wise clipping changes the direction
(large components are cut, small ones kept). Clipping at norm 1.0 is standard for [[RNN and LSTM]] and [[Transformers]] ([[Optimizers]]); logging `norm_before` is also a great diagnostic.

```python
def clip_global_norm(grads, max_norm):
    norm = np.sqrt(sum((g ** 2).sum() for g in grads))
    scale = min(1.0, max_norm / (norm + 1e-12))
    return [g * scale for g in grads], norm

grads_big = [rng.normal(size=(3, 4)) * 10, rng.normal(size=4) * 10]
grads_small = [g / 1000 for g in grads_big]
clipped_big, norm_big = clip_global_norm(grads_big, 1.0)
clipped_small, _ = clip_global_norm(grads_small, 1.0)
```

</details>

### Exercise 16 · Sanity check: overfit 10 examples
*💻 Code · ★★☆*

Checklist items 1 and 2. Using the ready-made `mlp_init` / `mlp_loss_grads` from the setup (64 tanh hidden units), take **10** digits from `load_digits`
(pixels / 16) and train with full-batch gradient descent (lr = 0.5) until the loss is near zero. Store the loss at step 0 in `loss0` and the final loss in `loss_final`.

In [ ]:
Xd, yd = load_digits(return_X_y=True)
X10, y10 = Xd[:10] / 16.0, yd[:10]
params = mlp_init(64, 64, 10)
loss0 = None
loss_final = None

check('initial loss ~ log 10', loss0, np.log(10), tol=0.05)
check('final loss < 0.01', None if loss_final is None else loss_final < 0.01, True)

<details>
<summary><b>💡 Hint</b></summary>

`for step in range(1000): loss, grads = mlp_loss_grads(params, X10, y10); params = [p - 0.5 * g for p, g in zip(params, grads)]`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The initial loss is $\approx\log10 = 2.30$ and the network memorises 10 examples within a few hundred steps. If this fails, there is a bug in the model, the loss or the data pipeline,
and no amount of tuning on the full dataset will help. Only after this test passes: train on all data, then add regularisation back (item 4).

```python
Xd, yd = load_digits(return_X_y=True)
X10, y10 = Xd[:10] / 16.0, yd[:10]
params = mlp_init(64, 64, 10)
for step in range(1000):
    loss, grads = mlp_loss_grads(params, X10, y10)
    if step == 0:
        loss0 = loss
    params = [p - 0.5 * g for p, g in zip(params, grads)]
loss_final = mlp_loss_grads(params, X10, y10)[0]
print(loss0, loss_final)
```

</details>

### Exercise 17 · Early stopping with patience
*💻 Code · ★★☆*

Implement `early_stopping(val_losses, patience, min_delta=0.0)`: walk through the validation losses epoch by epoch, track the best value
(an improvement must beat the best by more than `min_delta`), and stop once `patience` epochs pass without improvement.
Return `(best_epoch, stop_epoch)` (0-based; `stop_epoch` is the epoch at which training stops, or the last epoch if it never stops).

In [ ]:
def early_stopping(val_losses, patience, min_delta=0.0):
    return None, None

_e = np.arange(100)
val_curve = 1 / (1 + 0.1 * _e) + 0.004 * np.maximum(0, _e - 40) + 0.01 * np.random.default_rng(0).normal(size=100)

def _ref(v, patience, min_delta=0.0):
    best = np.inf; best_ep = 0
    for ep, x in enumerate(v):
        if x < best - min_delta:
            best, best_ep = x, ep
        elif ep - best_ep >= patience:
            return best_ep, ep
    return best_ep, len(v) - 1
_res = early_stopping(val_curve, 5)
check('patience 5', None if _res[0] is None else list(_res), list(_ref(val_curve, 5)))
_res = early_stopping(val_curve, 10, 0.005)
check('patience 10, min_delta', None if _res[0] is None else list(_res), list(_ref(val_curve, 10, 0.005)))
check('never stops on a decreasing curve', None if early_stopping(np.linspace(1, 0, 20), 3)[0] is None else list(early_stopping(np.linspace(1, 0, 20), 3)), [19, 19])

<details>
<summary><b>💡 Hint</b></summary>

Keep `best` and `best_epoch`; at each epoch either update them or check `epoch - best_epoch >= patience`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

After stopping, restore the weights saved at `best_epoch` (keep a checkpoint of the best model). Patience trades robustness to noisy validation curves against wasted epochs,
and `min_delta` ignores negligible improvements. Early stopping acts like a regulariser, limiting how far weights move from their init ([[Overfitting and Regularization]]).

```python
def early_stopping(val_losses, patience, min_delta=0.0):
    best, best_epoch = np.inf, 0
    for epoch, v in enumerate(val_losses):
        if v < best - min_delta:
            best, best_epoch = v, epoch
        elif epoch - best_epoch >= patience:
            return best_epoch, epoch
    return best_epoch, len(val_losses) - 1

_e = np.arange(100)
val_curve = 1 / (1 + 0.1 * _e) + 0.004 * np.maximum(0, _e - 40) + 0.01 * np.random.default_rng(0).normal(size=100)
print(early_stopping(val_curve, 5), early_stopping(val_curve, 10, 0.005))
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Training Tricks](Training%20Tricks.md).*